# forex_rl_v4 — Colab GPU training (multi-contributor)

Runs a slice of the full walk-forward sweep (12 folds x 3 seeds = 36
fold/seed combos, covering all 16 years of data on disk) as concurrent
`train.py` processes on ONE Colab GPU, writing to a Google Drive folder
SHARED across everyone contributing — so multiple people's sessions combine
into one pool of results instead of each person's work being stuck on
their own Drive.

**Before running:** `Runtime -> Change runtime type -> GPU`.

**Setup (do this once per person):**
1. Get `forex_rl_v4.zip` from the project owner and upload it to YOUR
   Google Drive as `MyDrive/forex_rl_v4.zip`.
2. Get the shared results folder's path from the owner (same for everyone).
3. Pick a `MY_NAME` in the config cell below and how many shards you want
   (`N_SHARDS_WANTED`, 4 is the suggested default) — the system claims
   free shard indices for you automatically, no manual range needed.

**Why code+data land on local disk, not Drive:** unzipping many files
(28 CSVs, 182MB) directly onto a Drive mount is flaky — Drive's FUSE layer
can throw `Operation not permitted` mid-write. `data/raw` is read-only and
doesn't need Drive's durability (it's just re-extracted from the zip in
seconds on a fresh session), so it's extracted to Colab's local disk
instead. Only `checkpoints/`+`runs/` need to survive a disconnect, so each
process writes those straight to the shared Drive folder — far less
file-churn than the CSV burst-write, much less likely to trip the same
issue.

**If the session disconnects:** re-run the notebook from the top. The claim
cell hands you back the SAME shard indices (keyed on `MY_NAME`), and each
process's `--resume` skips fold/seed combos already finished (by ANYONE,
not just you) and resumes an interrupted fold from its last mid-fold
checkpoint.</cell id="cell-0">


In [ ]:
# === Multi-contributor config — set these before running ===
# MY_NAME: anything that identifies YOU uniquely (first name is fine) —
# used as the key in the shared claims registry so the system can tell your
# shards apart from everyone else's and hand you the SAME ones back if you
# re-run this cell (e.g. after a disconnect) instead of grabbing new ones.
MY_NAME = 'change-me'

# N_SHARDS_WANTED: how many shard slots to claim this session. Up to 4 is a
# reasonable ceiling for one Colab GPU/CPU — check !nvidia-smi and !uptime
# after launching before going higher.
N_SHARDS_WANTED = 4

# TOTAL_SHARDS: total shard count across EVERYONE combined — the
# denominator for train.py's round-robin split. Must match what everyone
# else in the group is using. 36 = 12 folds x 3 seeds, i.e. one shard per
# fold/seed combo (the most parallelism this sweep can actually use —
# claiming more than that just means some shards find nothing left to do).
TOTAL_SHARDS = 36

# SHARED_FOLDER: the Drive folder the project owner shared with you.
# Everyone points at the SAME folder so results combine into one pool
# instead of scattering across separate accounts.
SHARED_FOLDER = '/content/drive/MyDrive/forex_rl_v4_shared_results'

# ITERATIONS: PPO iterations per fold/seed combo.
ITERATIONS = 100

assert MY_NAME != 'change-me', 'set MY_NAME to something that identifies you first'
print(f'{MY_NAME}: wants {N_SHARDS_WANTED} of {TOTAL_SHARDS} total shards, '
      f'{ITERATIONS} iterations/combo, writing to {SHARED_FOLDER}')</cell id="cell-1">


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Fail fast on a wrong/stale/incomplete zip instead of silently producing
# an empty data/raw later. Also auto-detects the zip's layout — some
# zip tools nest everything under a `forex_rl_v4/` folder, others (e.g.
# "compress selected files" in a file browser) put the contents directly at
# the zip's top level. Some Windows zip tools also write backslashes as the
# internal path separator instead of the ZIP spec's forward slash — normalize
# every entry name up front so neither convention needs special-casing below.
import os
import zipfile

ZIP_PATH = '/content/drive/MyDrive/forex_rl_v4.zip'

assert os.path.exists(ZIP_PATH), f'{ZIP_PATH} not found — upload forex_rl_v4.zip to MyDrive first.'

size_mb = os.path.getsize(ZIP_PATH) / 1e6
print(f'{ZIP_PATH}: {size_mb:.1f} MB')
assert size_mb > 20, (
    f'{ZIP_PATH} is only {size_mb:.1f} MB — too small to contain data/raw '
    f'(should be ~40+ MB). You likely uploaded a stale/code-only zip, or the '
    f'Drive upload did not finish. Re-upload the real forex_rl_v4.zip and re-run this cell.'
)

with zipfile.ZipFile(ZIP_PATH) as z:
    names = z.namelist()
    bad = z.testzip()
    norm_names = [n.replace('\\', '/') for n in names]  # normalize once, for every check below

    config_entries = [n for n in norm_names if n.endswith('config.py') and n.count('/') <= 1]
    assert config_entries, (
        f'{ZIP_PATH} has no config.py at its top level or one folder deep — '
        f'this is not the project zip. Re-upload the real forex_rl_v4.zip.'
    )
    ZIP_ROOT_PREFIX = config_entries[0].rsplit('config.py', 1)[0]  # '' or e.g. 'forex_rl_v4/'
    print(f'detected zip layout: files live under {ZIP_ROOT_PREFIX!r} inside the zip')

    csvs = [n for n in norm_names if n.startswith(f'{ZIP_ROOT_PREFIX}data/raw/') and n.endswith('.csv')]

assert bad is None, f'{ZIP_PATH} is corrupt (first bad file: {bad}) — re-upload it.'
assert len(csvs) >= 20, (
    f'{ZIP_PATH} only has {len(csvs)} CSV(s) under {ZIP_ROOT_PREFIX}data/raw/ (expected 28) — '
    f'wrong zip or an incomplete upload. Re-upload the real forex_rl_v4.zip.'
)
print(f'zip OK: {len(csvs)} CSVs found under {ZIP_ROOT_PREFIX}data/raw/')

In [10]:
# Extract to Colab's LOCAL disk (fast, reliable) — not the Drive mount.
# See the intro cell for why: unzipping many files onto Drive's FUSE layer
# is flaky ("Operation not permitted"), and data/raw is read-only so it
# doesn't need Drive's durability anyway. ONE shared copy — every parallel
# process reads the same code+data; only checkpoints/runs differ per shard
# (set up in the launch cell below via FOREX_RL_CHECKPOINT_DIR/RUNS_DIR).
#
# Handles either zip layout detected above: if the zip already nests
# everything under a folder (ZIP_ROOT_PREFIX != ''), extract straight to
# /content and rename that folder to PROJECT_DIR; if the zip has no
# top-level folder (ZIP_ROOT_PREFIX == ''), extract directly into PROJECT_DIR.
PROJECT_DIR = '/content/forex_rl_v4'

if not os.path.isdir(PROJECT_DIR):
    if ZIP_ROOT_PREFIX:
        !unzip -q "{ZIP_PATH}" -d /content
        # Fix: Strip both forward and backward slashes to get the correct directory name
        extracted_dir = f'/content/{ZIP_ROOT_PREFIX.rstrip('/\\')}'
        if extracted_dir != PROJECT_DIR:
            os.rename(extracted_dir, PROJECT_DIR)
    else:
        os.makedirs(PROJECT_DIR, exist_ok=True)
        !unzip -q "{ZIP_PATH}" -d "{PROJECT_DIR}"
else:
    print(f'{PROJECT_DIR} already exists — skipping unzip (safe to re-run this cell).')

%cd {PROJECT_DIR}

/content/forex_rl_v4 already exists — skipping unzip (safe to re-run this cell).
/content/forex_rl_v4


In [ ]:
# Claim your shard indices from the shared registry — the system figures
# out which ones are still free, you don't need to be handed a range
# manually. Safe to re-run: idempotent for the SAME MY_NAME (returns your
# existing claims first, only grabs new ones for any shortfall).
import sys
sys.path.insert(0, PROJECT_DIR)
from scripts.claim_shards import claim

MY_SHARDS = claim(SHARED_FOLDER, TOTAL_SHARDS, N_SHARDS_WANTED, MY_NAME)
print(f'{MY_NAME}: claimed shards {MY_SHARDS}')

In [11]:
# MetaTrader5 is Windows-only and gated by an environment marker in
# requirements.txt (`; platform_system == "Windows"`) — pip skips it
# automatically here, and nothing in the training path imports it.
!pip install -q -r requirements.txt

In [12]:
import torch
assert torch.cuda.is_available(), (
    "No GPU visible. Runtime -> Change runtime type -> GPU, then re-run this cell."
)
print('GPU:', torch.cuda.get_device_name(0))

GPU: Tesla T4


In [13]:
# Sanity check first — same coverage report you'd run locally.
!python data_pipeline.py

Found data for 28/28 pairs in /content/forex_rl_v4/data/raw

  EURUSD: from 2010-08-24, 99.1% coverage
  GBPUSD: from 2010-08-24, 99.1% coverage
  USDJPY: from 2010-08-24, 99.2% coverage
  USDCHF: from 2010-08-24, 99.2% coverage
  AUDUSD: from 2010-08-24, 99.2% coverage
  USDCAD: from 2010-08-24, 99.2% coverage
  NZDUSD: from 2010-08-24, 99.1% coverage
  EURGBP: from 2010-08-24, 99.2% coverage
  EURJPY: from 2010-08-24, 99.1% coverage
  EURCHF: from 2010-08-24, 98.9% coverage
  EURAUD: from 2010-08-24, 99.2% coverage
  EURCAD: from 2010-08-24, 99.2% coverage
  EURNZD: from 2010-08-24, 99.1% coverage
  GBPJPY: from 2010-08-24, 99.2% coverage
  GBPCHF: from 2010-08-24, 99.1% coverage
  GBPAUD: from 2010-08-24, 99.1% coverage
  GBPCAD: from 2010-08-24, 99.2% coverage
  GBPNZD: from 2010-08-24, 96.6% coverage
  AUDJPY: from 2010-08-24, 99.1% coverage
  AUDCHF: from 2010-08-24, 99.2% coverage
  AUDCAD: from 2010-08-24, 99.2% coverage
  AUDNZD: from 2010-08-24, 99.1% coverage
  CADJPY: from 

## Optional: bump ROLLOUT_N_ENVS for GPU

`config.ROLLOUT_N_ENVS` (default 32) sets how many environments are batched
into one model forward call. The model is small (2-5M params) and Colab GPUs
have plenty of headroom, so a higher value amortizes kernel-launch overhead
further. Edit `config.py` directly (or uncomment below) if you want to try
64 or 128 — there's no single right answer, and changing it does change the
exact (though not the statistical) outcome for a given seed, so pick a value
and keep it fixed for the whole sweep.

In [ ]:
# import config
# print('current ROLLOUT_N_ENVS:', config.ROLLOUT_N_ENVS)
# Edit config.py's ROLLOUT_N_ENVS value directly instead of monkey-patching
# here, since train.py runs as a subprocess below and won't see an in-notebook
# variable change.

In [ ]:
import subprocess
import time

# Each of YOUR processes gets its own subfolder under the SHARED Drive
# folder (via env var overrides — see config.py) so concurrent processes —
# yours or anyone else's, in case sessions overlap — never read-modify-write
# the SAME RUN_MANIFEST.json at once. All processes share the one local
# code+data checkout (PROJECT_DIR) — data/raw is read-only.
os.makedirs(SHARED_FOLDER, exist_ok=True)

procs = []
for idx in MY_SHARDS:
    shard_dir = f'{SHARED_FOLDER}/shard{idx}'
    ckpt_dir = f'{shard_dir}/checkpoints'
    runs_dir = f'{shard_dir}/runs'
    os.makedirs(ckpt_dir, exist_ok=True)
    os.makedirs(runs_dir, exist_ok=True)

    env = os.environ.copy()
    env['FOREX_RL_CHECKPOINT_DIR'] = ckpt_dir
    env['FOREX_RL_RUNS_DIR'] = runs_dir

    log_path = f'{shard_dir}/train_log.txt'
    p = subprocess.Popen(
        ['python', 'train.py', '--resume', '--iterations', str(ITERATIONS),
         '--shard-index', str(idx), '--shard-count', str(TOTAL_SHARDS)],
        cwd=PROJECT_DIR, env=env,
        stdout=open(log_path, 'w'), stderr=subprocess.STDOUT,
    )
    procs.append({'idx': idx, 'proc': p, 'log': log_path})
    print(f'started shard {idx} (pid {p.pid}) -> {log_path}')
    time.sleep(2)  # stagger starts slightly so they don't all hit disk/data-loading at once

print(f'\n{len(procs)} shard(s) running in the background. Use the next cell '
      f'(re-run it any time) to check progress.')</cell id="cell-11">


In [22]:
# Re-run this cell any time to check progress. All "exited (code 0)" means
# every shard finished — move on to the merge cell at the bottom.
for entry in procs:
    rc = entry['proc'].poll()
    status = 'running' if rc is None else f'exited (code {rc})'
    print(f"shard {entry['idx']} (pid {entry['proc'].pid}): {status}")
    !tail -n 3 "{entry['log']}"
    print()

# Resource check — see the parallelism config cell for what to watch for.
!nvidia-smi --query-gpu=memory.used,memory.total,utilization.gpu --format=csv
!uptime

shard 0 (pid 3134): exited (code -9)
2026-08-25 03:57:25,461 [__main__] INFO: === fold 0 / seed 13 ===
2026-08-25 03:57:25,470 [fold_runner] INFO: [fold 0] loading edge features
2026-08-25 03:59:22,622 [fold_runner] INFO: [fold 0] starting PPO training (50 iterations, 32 rollout lanes)

shard 1 (pid 3149): running
2026-08-25 03:57:36,706 [__main__] INFO: === fold 0 / seed 42 ===
2026-08-25 03:57:36,713 [fold_runner] INFO: [fold 0] loading edge features
2026-08-25 03:59:22,627 [fold_runner] INFO: [fold 0] starting PPO training (50 iterations, 32 rollout lanes)

shard 2 (pid 3161): exited (code -9)
2026-08-25 03:57:47,075 [__main__] INFO: === fold 0 / seed 7 ===
2026-08-25 03:57:47,084 [fold_runner] INFO: [fold 0] loading edge features
2026-08-25 03:59:22,627 [fold_runner] INFO: [fold 0] starting PPO training (50 iterations, 32 rollout lanes)

shard 3 (pid 3177): running
2026-08-25 03:57:51,471 [__main__] INFO: === fold 1 / seed 13 ===
2026-08-25 03:57:51,479 [fold_runner] INFO: [fold 1]

## Merge and archive

Every contributor's process wrote to their own `shard{i}/` subfolder under
the ONE shared Drive folder. This pulls in ALL `TOTAL_SHARDS` of them —
everyone's combined progress, not just yours — into one consolidated
`checkpoints/`+`runs/` in `PROJECT_DIR`, and zips that up for download.
Safe to run any time, by anyone with access to the shared folder, even
mid-training.</cell id="cell-12">


In [ ]:
import datetime

all_shard_dirs = ' '.join(f'{SHARED_FOLDER}/shard{idx}' for idx in range(TOTAL_SHARDS))
!python scripts/merge_shard_results.py {all_shard_dirs}

stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
archive_path = f'/content/drive/MyDrive/forex_rl_v4_merged_results_{stamp}.zip'
!zip -rq "{archive_path}" checkpoints runs
print('wrote', archive_path)</cell id="cell-13">


## Live dashboard (optional)

Runs the project's dashboard (Training / Performance / Price chart / Agent
replay / Compare seeds) right here on Colab, exposed via a Cloudflare quick
tunnel — a public URL you can open from any device, no local PC needed. No
account or token required; the URL only exists for this session and stops
working once the tunnel process ends.

Safe to run this at ANY point, including while the shards above are still
training — it merges whatever progress exists so far (additive, not
destructive) so the dashboard has something to show immediately. To refresh
with newer progress later, either re-run this cell, or use the dashboard's
own "Merge shard results" section (Training page) with the shard paths this
cell prints — no need to come back to the notebook at all once it's open.</cell id="cell-13">


In [ ]:
import os
import re
import subprocess
import time

# 1. Pull in EVERY contributor's progress so far — safe mid-training,
# merge_shard_results.py is additive (dedupes by fold_id/seed/config, never
# deletes). Populates PROJECT_DIR/checkpoints+runs so the dashboard has
# something to show immediately.
shard_dirs = [f'{SHARED_FOLDER}/shard{idx}' for idx in range(TOTAL_SHARDS)]
print('Merging current progress from:')
for d in shard_dirs:
    print(' ', d)
!python scripts/merge_shard_results.py {' '.join(shard_dirs)}

# 2. Install cloudflared once per session (small binary, local disk is fine —
# it's a downloaded tool, not something that needs to survive a disconnect).
if not os.path.exists('/content/cloudflared'):
    print('\ninstalling cloudflared...')
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
    !chmod +x /content/cloudflared

# 3. Launch Streamlit in the background — skip if a previous run of this
# cell already has it up, so re-running just re-merges + re-prints the URL
# instead of double-launching and fighting over port 8501.
if 'streamlit_proc' in dir() and streamlit_proc.poll() is None:
    print(f'\nStreamlit already running (pid {streamlit_proc.pid}) — not relaunching.')
else:
    streamlit_log = '/content/streamlit_log.txt'
    streamlit_proc = subprocess.Popen(
        ['streamlit', 'run', 'dashboard/streamlit_app.py', '--server.port', '8501', '--server.headless', 'true'],
        cwd=PROJECT_DIR, stdout=open(streamlit_log, 'w'), stderr=subprocess.STDOUT,
    )
    print(f'\nstarted Streamlit (pid {streamlit_proc.pid}), waiting for it to boot...')
    time.sleep(8)

# 4. Launch the tunnel (same re-run guard) and pull its public URL from the log.
if 'tunnel_proc' in dir() and tunnel_proc.poll() is None:
    print(f'Tunnel already running (pid {tunnel_proc.pid}) — reusing existing URL below.')
else:
    tunnel_log = '/content/cloudflared_log.txt'
    tunnel_proc = subprocess.Popen(
        ['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
        stdout=open(tunnel_log, 'w'), stderr=subprocess.STDOUT,
    )
    time.sleep(6)

with open(tunnel_log) as f:
    log_text = f.read()
match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', log_text)
if match:
    print('\nDashboard URL:', match.group(0))
    print('(open from any device — stays live as long as this cell keeps running)')
else:
    print('\nTunnel URL not in the log yet — wait a few seconds and re-run just this cell.')
    print('If it keeps happening, check the raw log:', tunnel_log)

print('\nThis dashboard already shows EVERYONE\'s combined progress (it merged all')
print('TOTAL_SHARDS above). Re-run this cell any time to refresh with newer progress.')</cell id="cell-15">
